# Lab 4: Trenowanie modelu ML i serwowanie przez FastAPI

## Cel

- Wytrenowanie modelu detekcji fraudów offline,
- Serwowanie modelu przez FastAPI,
- Połączenie API z konsumentem Kafki.

**Case biznesowy (ciąg dalszy):** Reguły działają, ale chcesz poprawić dokładność modelem ML.

---

## Część 1: Trenowanie modelu

### Zadanie 1.1 — Przygotuj dane treningowe

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import pickle

np.random.seed(42)
n_normal, n_fraud = 2000, 100

normal = pd.DataFrame({
    'amount': np.random.lognormal(5, 1, n_normal).clip(5, 5000),
    'hour': np.random.normal(14, 4, n_normal).clip(0, 23).astype(int),
    'is_electronics': np.random.binomial(1, 0.3, n_normal),
    'tx_per_day': np.random.poisson(3, n_normal),
    'fraud': 0
})

fraud = pd.DataFrame({
    'amount': np.random.uniform(2000, 9000, n_fraud),
    'hour': np.random.choice([0,1,2,3,4,5,22,23], n_fraud),
    'is_electronics': np.random.binomial(1, 0.7, n_fraud),
    'tx_per_day': np.random.poisson(8, n_fraud),
    'fraud': 1
})

df = pd.concat([normal, fraud], ignore_index=True).sample(frac=1, random_state=42)
print(f"Dane: {len(df)} wierszy, fraud rate: {df['fraud'].mean():.1%}")

### Zadanie 1.2 — Wytrenuj i oceń model

In [ ]:
features = ['amount', 'hour', 'is_electronics', 'tx_per_day']
# TWÓJ KOD
# 1. Podziel dane (80/20, stratify)
# 2. Wytrenuj RandomForestClassifier
# 3. Wyświetl classification_report
# 4. Zapisz model do 'fraud_model.pkl'

---

## Część 2: API z FastAPI

### Zadanie 2.1 — Utwórz API

In [ ]:
%%file fraud_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle, numpy as np

app = FastAPI(title="Fraud Detection API")
model = pickle.load(open('fraud_model.pkl', 'rb'))

class Transaction(BaseModel):
    amount: float
    hour: int
    is_electronics: int
    tx_per_day: int

# TWÓJ KOD
# Endpoint POST /score

Uruchom: `uvicorn fraud_api:app --host 0.0.0.0 --port 8001`

### Zadanie 2.2 — Przetestuj API

In [ ]:
import requests
normal_tx = {"amount": 150, "hour": 14, "is_electronics": 0, "tx_per_day": 3}
response = requests.post("http://localhost:8001/score", json=normal_tx)
print("Normalna:", response.json())

# TWÓJ KOD — przetestuj podejrzaną transakcję

---

## Część 3: Połączenie API z Kafką

### Zadanie 3.1 — Konsument scoringowy z API

In [ ]:
%%file ml_consumer.py
from kafka import KafkaConsumer, KafkaProducer
import json, requests

# TWÓJ KOD
# Dla każdej transakcji: wyciągnij cechy, wywołaj API, wyślij alert jeśli fraud

---

## Praca domowa

1. Dodaj endpoint `/health` z metadanymi modelu.
2. Utwórz Dockerfile dla API.
3. Porównaj reguły (Lab 3) vs ML — co lepiej wykrywa fraudy?
4. Wypchnij do Git.

**Następne zajęcia:** Uczenie przyrostowe — model, który się aktualizuje.